In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Drag $f_m$ from a slow wobble up to an audible rate. On the left, the
gray outline is the envelope the modulator traces around the carrier. On
the right, the two teal sidebands slide apart from the dashed carrier at
240 Hz. The audio card plays the current setting, so you can hear the
moment one pulsing tone becomes two steady ones.

In [ ]:
# hide
# autorun
FC, FM0 = 240.0, 3.0                # the chapter's carrier, and a slow start

T_WIN = 0.5                         # half a second on screen
t = np.linspace(0.0, T_WIN, 5000)
CARRIER = np.sin(2 * np.pi * FC * t)
SR = 44100
T_PLAY = np.arange(int(2.5 * SR)) / SR
CARRIER_P = np.sin(2 * np.pi * FC * T_PLAY)

def figure():
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.13)
    env = np.abs(np.sin(2 * np.pi * FM0 * t))
    fig.add_scatter(x=t * 1000, y=CARRIER * np.sin(2 * np.pi * FM0 * t),
                    mode="lines", line=dict(color=RED, width=1.0),
                    row=1, col=1)
    for sign in (1.0, -1.0):      # the outline sits on top of the band
        fig.add_scatter(x=t * 1000, y=sign * env, mode="lines",
                        line=dict(color=IRON, width=1.8, dash="dash"),
                        row=1, col=1)
    fig.add_scatter(x=[FC, FC, None], y=[0, 1.0, None], mode="lines",
                    line=dict(color=RED, width=1.4, dash="dot"), row=1, col=2)
    fig.add_scatter(x=[FM0, FM0, None], y=[0, 1.0, None], mode="lines",
                    line=dict(color=BLUE, width=1.4, dash="dot"),
                    row=1, col=2)
    fig.add_scatter(x=[FC - FM0, FC - FM0, None, FC + FM0, FC + FM0, None],
                    y=[0, 0.5, None, 0, 0.5, None], mode="lines",
                    line=dict(color=TEAL, width=3.4), row=1, col=2)
    fig.update_xaxes(range=[0, T_WIN * 1000], title_text="Time (ms)",
                     fixedrange=True, row=1, col=1)
    fig.update_yaxes(range=[-1.15, 1.15], title_text="Amplitude",
                     fixedrange=True, row=1, col=1)
    fig.update_xaxes(range=[0, 420], title_text="Frequency (Hz)",
                     fixedrange=True, row=1, col=2)
    fig.update_yaxes(range=[0, 1.08], title_text="Magnitude",
                     fixedrange=True, row=1, col=2)
    return fig

def controls(fig):
    fm = widgets.FloatSlider(description="Modulator f_m (Hz)", min=2.0,
                             max=120.0, value=FM0, step=0.5)
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(fm, t=t, CARRIER=CARRIER, FC=FC, readout=readout):
        mod = np.sin(2 * np.pi * fm * t)
        env = np.abs(mod)
        with fig.batch_update():
            fig.data[0].y = CARRIER * mod
            fig.data[1].y = env
            fig.data[2].y = -env
            fig.data[4].x = [fm, fm, None]
            fig.data[5].x = [FC - fm, FC - fm, None, FC + fm, FC + fm, None]
        readout.value = (f"<span style='font-size:0.9em'>sidebands at "
                         f"{FC - fm:.1f} Hz and {FC + fm:.1f} Hz "
                         f"&nbsp;·&nbsp; {2 * fm:.1f} swells per second"
                         f"</span>")

    widgets.interactive_output(update, {"fm": fm})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def render(T_PLAY=T_PLAY, CARRIER_P=CARRIER_P, SR=SR):
        x = 0.125 * CARRIER_P * np.sin(2 * np.pi * fm.value * T_PLAY)
        x[:441] *= np.linspace(0, 1, 441)
        x[-441:] *= np.linspace(1, 0, 441)
        audio = Audio(x.astype(np.float32), rate=SR, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)


    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (fm,):
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([fm, readout, out, gate])

icm_plotly.show(figure, controls)